In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

project_path = Path.cwd()
data_path = project_path / "data"
processed_path = data_path / "processed"

print("Phase 13 - Advanced Analytics")
print("Project path:", project_path)
print("Processed folder exists:", processed_path.exists())

Phase 13 - Advanced Analytics
Project path: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets
Processed folder exists: True


In [3]:
# Load cleaned operational datasets

baggage = pd.read_csv(
    processed_path / "baggage_clean.csv"
)

flight = pd.read_csv(
    processed_path / "flight_clean.csv"
)

baggage_scan_event = pd.read_csv(
    processed_path / "baggage_scan_event_clean.csv"
)

# Load investigation and exception datasets

advanced_investigation = pd.read_csv(
    processed_path / "final_phase11_advanced_investigation.csv"
)

exceptions = pd.read_csv(
    processed_path / "phase12_exception_records.csv"
)

print("ALL ANALYTICS DATASETS LOADED SUCCESSFULLY")

print("\nBaggage:", baggage.shape)
print("Flight:", flight.shape)
print("Scan Events:", baggage_scan_event.shape)
print("Advanced Investigation:", advanced_investigation.shape)
print("Exception Records:", exceptions.shape)

ALL ANALYTICS DATASETS LOADED SUCCESSFULLY

Baggage: (1000, 5)
Flight: (250, 6)
Scan Events: (9000, 9)
Advanced Investigation: (1000, 33)
Exception Records: (685, 12)


In [4]:
# Convert operational timestamps

baggage_scan_event["event_time"] = pd.to_datetime(
    baggage_scan_event["event_time"],
    utc=True,
    errors="coerce"
)

baggage_scan_event["received_at"] = pd.to_datetime(
    baggage_scan_event["received_at"],
    utc=True,
    errors="coerce"
)

baggage["created_at"] = pd.to_datetime(
    baggage["created_at"],
    utc=True,
    errors="coerce"
)

print("Timestamp conversion completed!")

print("\nScan event timestamp types:")
print(baggage_scan_event[["event_time", "received_at"]].dtypes)

print("\nBaggage timestamp type:")
print(baggage["created_at"].dtype)

Timestamp conversion completed!

Scan event timestamp types:
event_time     datetime64[us, UTC]
received_at    datetime64[us, UTC]
dtype: object

Baggage timestamp type:
datetime64[us, UTC]


In [5]:
# Task 1: Baggage Journey Analytics

total_baggage = len(baggage)

journey_distribution = (
    advanced_investigation["journey_type"]
    .value_counts()
    .reset_index()
)

journey_distribution.columns = [
    "journey_type",
    "baggage_count"
]

journey_distribution["percentage"] = (
    journey_distribution["baggage_count"]
    / total_baggage
) * 100

print("TOTAL BAGGAGE:", total_baggage)

print("\nBAGGAGE JOURNEY DISTRIBUTION:")
display(journey_distribution)

TOTAL BAGGAGE: 1000

BAGGAGE JOURNEY DISTRIBUTION:


,journey_type,baggage_count,percentage
0,DIRECT,500,50.0
1,CONNECTING,500,50.0


In [6]:
# Investigation status by journey type

journey_status_analysis = pd.crosstab(
    advanced_investigation["journey_type"],
    advanced_investigation["advanced_detection_status"]
)

print("INVESTIGATION STATUS BY JOURNEY TYPE:")
display(journey_status_analysis)

INVESTIGATION STATUS BY JOURNEY TYPE:


advanced_detection_status,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,MISSING_EVENT_ONLY,NO_ADVANCED_ISSUE_DETECTED
journey_type,,,
CONNECTING,185,315,0
DIRECT,0,0,500


In [7]:
# Calculate investigation rate by journey type

journey_investigation = (
    advanced_investigation
    .groupby("journey_type")
    .agg(
        total_baggage=("bag_id", "nunique"),
        baggage_with_issues=("advanced_issue_detected", "sum")
    )
    .reset_index()
)

journey_investigation["investigation_rate"] = (
    journey_investigation["baggage_with_issues"]
    / journey_investigation["total_baggage"]
) * 100

print("JOURNEY-WISE INVESTIGATION RATE:")
display(journey_investigation)

JOURNEY-WISE INVESTIGATION RATE:


,journey_type,total_baggage,baggage_with_issues,investigation_rate
0,CONNECTING,500,500,100.0
1,DIRECT,500,0,0.0


In [8]:
# Exception severity analysis

severity_analysis = (
    exceptions
    .groupby("severity")
    .agg(
        total_exceptions=("exception_id", "count"),
        unique_baggage=("bag_id", "nunique")
    )
    .reset_index()
)

print("EXCEPTION SEVERITY ANALYSIS:")
display(severity_analysis)

EXCEPTION SEVERITY ANALYSIS:


,severity,total_exceptions,unique_baggage
0,HIGH,185,185
1,MEDIUM,500,500


In [9]:
# Exception type analysis

exception_type_analysis = (
    exceptions
    .groupby(["exception_type", "severity"])
    .agg(
        total_exceptions=("exception_id", "count"),
        unique_baggage=("bag_id", "nunique")
    )
    .reset_index()
)

print("EXCEPTION TYPE ANALYSIS:")
display(exception_type_analysis)

EXCEPTION TYPE ANALYSIS:


,exception_type,severity,total_exceptions,unique_baggage
0,HANDOFF_TIMING_INCONSISTENCY,HIGH,185,185
1,MISSING_SEGMENT_EVENT,MEDIUM,500,500


In [10]:
# Exception lifecycle analysis

exception_lifecycle = (
    exceptions
    .groupby(["status", "severity"])
    .agg(
        total_exceptions=("exception_id", "count"),
        unique_baggage=("bag_id", "nunique")
    )
    .reset_index()
)

print("EXCEPTION LIFECYCLE ANALYSIS:")
display(exception_lifecycle)

EXCEPTION LIFECYCLE ANALYSIS:


,status,severity,total_exceptions,unique_baggage
0,OPEN,HIGH,184,184
1,OPEN,MEDIUM,500,500
2,RESOLVED,HIGH,1,1


In [11]:
# Overall exception KPIs

total_exceptions = len(exceptions)

open_exceptions = (
    exceptions["status"] == "OPEN"
).sum()

in_progress_exceptions = (
    exceptions["status"] == "IN_PROGRESS"
).sum()

resolved_exceptions = (
    exceptions["status"] == "RESOLVED"
).sum()

high_priority_exceptions = (
    exceptions["severity"] == "HIGH"
).sum()

medium_priority_exceptions = (
    exceptions["severity"] == "MEDIUM"
).sum()

exception_kpis = pd.DataFrame({
    "metric": [
        "Total Exceptions",
        "Open Exceptions",
        "In Progress Exceptions",
        "Resolved Exceptions",
        "High Priority Exceptions",
        "Medium Priority Exceptions"
    ],
    "value": [
        total_exceptions,
        open_exceptions,
        in_progress_exceptions,
        resolved_exceptions,
        high_priority_exceptions,
        medium_priority_exceptions
    ]
})

print("EXCEPTION KPI SUMMARY:")
display(exception_kpis)

EXCEPTION KPI SUMMARY:


,metric,value
0,Total Exceptions,685
1,Open Exceptions,684
2,In Progress Exceptions,0
3,Resolved Exceptions,1
4,High Priority Exceptions,185
5,Medium Priority Exceptions,500


In [12]:
# Airport-wise scan activity

airport_scan_analysis = (
    baggage_scan_event
    .groupby("airport_code")
    .agg(
        total_scan_events=("event_id", "count"),
        unique_baggage=("bag_id", "nunique"),
        event_types=("event_type", "nunique")
    )
    .reset_index()
    .sort_values(
        by="total_scan_events",
        ascending=False
    )
)

print("AIRPORT-WISE SCAN ACTIVITY:")
display(airport_scan_analysis)

AIRPORT-WISE SCAN ACTIVITY:


,airport_code,total_scan_events,unique_baggage,event_types
8,MAA,1676,419,4
7,LHR,1317,439,3
4,DXB,1088,290,8
2,DEL,1065,278,8
3,DOH,1033,268,8
9,SIN,849,232,8
5,FRA,660,206,7
1,BOM,624,156,4
6,JFK,480,160,3
0,BLR,208,52,4


In [13]:
# Airport-wise event type distribution

airport_event_distribution = pd.crosstab(
    baggage_scan_event["airport_code"],
    baggage_scan_event["event_type"]
)

print("AIRPORT-WISE EVENT TYPE DISTRIBUTION:")
display(airport_event_distribution)

AIRPORT-WISE EVENT TYPE DISTRIBUTION:


event_type,ARRIVED,CHECK_IN,DELIVERED,LOADED,SECURITY_SCAN,SORTED,TRANSFER,UNLOADED
airport_code,,,,,,,,
BLR,0,52,0,52,52,52,0,0
BOM,0,156,0,156,156,156,0,0
DEL,124,154,47,231,154,154,77,124
DOH,223,45,39,229,45,45,184,223
DXB,195,95,72,218,95,95,123,195
FRA,164,42,164,42,42,42,0,164
JFK,160,0,160,0,0,0,0,160
LHR,439,0,439,0,0,0,0,439
MAA,0,419,0,419,419,419,0,0


In [14]:
# Airport-wise event percentages

airport_event_percentage = (
    pd.crosstab(
        baggage_scan_event["airport_code"],
        baggage_scan_event["event_type"],
        normalize="index"
    ) * 100
)

airport_event_percentage = airport_event_percentage.round(2)

print("AIRPORT-WISE EVENT PERCENTAGES:")
display(airport_event_percentage)

AIRPORT-WISE EVENT PERCENTAGES:


event_type,ARRIVED,CHECK_IN,DELIVERED,LOADED,SECURITY_SCAN,SORTED,TRANSFER,UNLOADED
airport_code,,,,,,,,
BLR,0.00,25.00,0.00,25.00,25.00,25.00,0.00,0.00
BOM,0.00,25.00,0.00,25.00,25.00,25.00,0.00,0.00
DEL,11.64,14.46,4.41,21.69,14.46,14.46,7.23,11.64
DOH,21.59,4.36,3.78,22.17,4.36,4.36,17.81,21.59
DXB,17.92,8.73,6.62,20.04,8.73,8.73,11.31,17.92
FRA,24.85,6.36,24.85,6.36,6.36,6.36,0.00,24.85
JFK,33.33,0.00,33.33,0.00,0.00,0.00,0.00,33.33
LHR,33.33,0.00,33.33,0.00,0.00,0.00,0.00,33.33
MAA,0.00,25.00,0.00,25.00,25.00,25.00,0.00,0.00


In [16]:
# Load cleaned baggage assignment dataset

baggage_assignment = pd.read_csv(
    processed_path / "baggage_assignment_clean.csv"
)

print("Baggage assignments loaded successfully!")
print("Shape:", baggage_assignment.shape)
display(baggage_assignment.head())

Baggage assignments loaded successfully!
Shape: (1500, 5)


,assignment_id,bag_id,flight_id,segment_sequence,assigned_at
0,ASG00001,BAG00001,FL0051,1,2026-11-01 10:00:00+00:00
1,ASG00002,BAG00002,FL0102,1,2026-11-02 01:00:00+00:00
2,ASG00003,BAG00003,FL0103,1,2026-11-03 01:00:00+00:00
3,ASG00004,BAG00004,FL0044,1,2026-11-04 01:00:00+00:00
4,ASG00005,BAG00005,FL0025,1,2026-11-05 01:00:00+00:00


In [17]:
# Flight-wise baggage assignment analysis

flight_assignment_analysis = (
    baggage_assignment
    .groupby("flight_id")
    .agg(
        total_baggage=("bag_id", "nunique"),
        total_assignments=("assignment_id", "count")
    )
    .reset_index()
)

print("FLIGHT-WISE BAGGAGE ASSIGNMENT:")
display(flight_assignment_analysis.head(10))

print("\nTotal flights with baggage assignments:",
      flight_assignment_analysis["flight_id"].nunique())

print("Total baggage assignments:",
      flight_assignment_analysis["total_assignments"].sum())

FLIGHT-WISE BAGGAGE ASSIGNMENT:


,flight_id,total_baggage,total_assignments
0,FL0001,7,7
1,FL0002,9,9
2,FL0003,10,10
3,FL0004,10,10
4,FL0005,6,6
5,FL0006,13,13
6,FL0007,9,9
7,FL0008,8,8
8,FL0009,6,6
9,FL0010,11,11



Total flights with baggage assignments: 179
Total baggage assignments: 1500


In [18]:
# Top 10 busiest flights by baggage assignments

top_flights = (
    flight_assignment_analysis
    .sort_values(
        by="total_baggage",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

print("TOP 10 BUSIEST FLIGHTS:")
display(top_flights)

TOP 10 BUSIEST FLIGHTS:


,flight_id,total_baggage,total_assignments
0,FL0051,19,19
1,FL0041,19,19
2,FL0101,17,17
3,FL0102,17,17
4,FL0057,16,16
5,FL0092,16,16
6,FL0020,16,16
7,FL0047,15,15
8,FL0079,15,15
9,FL0066,15,15


In [19]:
# Join flight assignments with flight details

flight_enriched = flight_assignment_analysis.merge(
    flight[
        ["flight_id", "flight_number", "source", "destination"]
    ],
    on="flight_id",
    how="left"
)

flight_enriched = flight_enriched[
    [
        "flight_id",
        "flight_number",
        "source",
        "destination",
        "total_baggage",
        "total_assignments"
    ]
]

print("ENRICHED FLIGHT ANALYTICS:")
display(flight_enriched.head(10))

print("\nMissing flight details:",
      flight_enriched["flight_number"].isna().sum())

ENRICHED FLIGHT ANALYTICS:


,flight_id,flight_number,source,destination,total_baggage,total_assignments
0,FL0001,BA1924,MAA,DXB,7,7
1,FL0002,AI4606,MAA,DXB,9,9
2,FL0003,EK3757,MAA,DXB,10,10
3,FL0004,EK1779,MAA,DXB,10,10
4,FL0005,BA9035,MAA,DXB,6,6
5,FL0006,AI9774,MAA,DXB,13,13
6,FL0007,SQ620,MAA,DXB,9,9
7,FL0008,AI1635,MAA,DXB,8,8
8,FL0009,EK3911,MAA,DXB,6,6
9,FL0010,LH9963,MAA,DXB,11,11



Missing flight details: 0


In [20]:
# Route-wise baggage analytics

route_analysis = (
    flight_enriched
    .groupby(["source", "destination"])
    .agg(
        total_flights=("flight_id", "nunique"),
        total_baggage=("total_baggage", "sum")
    )
    .reset_index()
    .sort_values(
        by="total_baggage",
        ascending=False
    )
)

print("ROUTE-WISE BAGGAGE ANALYTICS:")
display(route_analysis)

ROUTE-WISE BAGGAGE ANALYTICS:


,source,destination,total_flights,total_baggage
12,MAA,DEL,10,124
4,DEL,LHR,10,121
6,DOH,FRA,10,112
9,DXB,JFK,10,110
10,DXB,LHR,10,108
13,MAA,DOH,10,106
2,BOM,DXB,10,106
17,SIN,LHR,10,101
15,MAA,SIN,10,100
14,MAA,DXB,10,89


In [21]:
# Flight segments per baggage journey

segment_analysis = (
    baggage_assignment
    .groupby("bag_id")
    .agg(
        total_flight_segments=("flight_id", "nunique")
    )
    .reset_index()
)

segment_distribution = (
    segment_analysis["total_flight_segments"]
    .value_counts()
    .sort_index()
    .reset_index()
)

segment_distribution.columns = [
    "flight_segments",
    "baggage_count"
]

print("BAGGAGE FLIGHT SEGMENT DISTRIBUTION:")
display(segment_distribution)

BAGGAGE FLIGHT SEGMENT DISTRIBUTION:


,flight_segments,baggage_count
0,1,500
1,2,500


In [22]:
# Connecting journey investigation analysis

connecting_analysis = advanced_investigation[
    advanced_investigation["journey_type"] == "CONNECTING"
].copy()

connecting_kpis = pd.DataFrame({
    "metric": [
        "Total Connecting Baggage",
        "Missing Segment Event",
        "Handoff Timing Inconsistency",
        "Airport Consistency Issues",
        "Event Duration Pattern Issues"
    ],
    "value": [
        connecting_analysis["bag_id"].nunique(),
        connecting_analysis["missing_segment_event_detected"].sum(),
        connecting_analysis["handoff_timing_inconsistency_detected"].sum(),
        connecting_analysis["airport_consistency_issue_detected"].sum(),
        connecting_analysis["event_duration_pattern_detected"].sum()
    ]
})

print("CONNECTING JOURNEY INVESTIGATION KPIs:")
display(connecting_kpis)

CONNECTING JOURNEY INVESTIGATION KPIs:


,metric,value
0,Total Connecting Baggage,500
1,Missing Segment Event,500
2,Handoff Timing Inconsistency,185
3,Airport Consistency Issues,0
4,Event Duration Pattern Issues,0


In [23]:
# Handoff performance analytics

handoff_metrics = connecting_analysis[
    [
        "unload_to_transfer_minutes",
        "transfer_to_load_minutes"
    ]
].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print("HANDOFF PERFORMANCE METRICS:")
display(handoff_metrics)

HANDOFF PERFORMANCE METRICS:


,unload_to_transfer_minutes,transfer_to_load_minutes
count,500.0,500.0
mean,30.0,-129.6
median,30.0,270.0
min,30.0,-810.0
max,30.0,270.0


In [24]:
# Classify transfer-to-load chronology

connecting_analysis["handoff_chronology_category"] = np.where(
    connecting_analysis["transfer_to_load_minutes"] < 0,
    "INVALID_CHRONOLOGY",
    "POSITIVE_HANDOFF_GAP"
)

handoff_chronology_analysis = (
    connecting_analysis
    .groupby("handoff_chronology_category")
    .agg(
        baggage_count=("bag_id", "nunique"),
        average_gap_minutes=("transfer_to_load_minutes", "mean")
    )
    .reset_index()
)

handoff_chronology_analysis["percentage"] = (
    handoff_chronology_analysis["baggage_count"] / 500
) * 100

print("HANDOFF CHRONOLOGY CLASSIFICATION:")
display(handoff_chronology_analysis.round(2))

HANDOFF CHRONOLOGY CLASSIFICATION:


,handoff_chronology_category,baggage_count,average_gap_minutes,percentage
0,INVALID_CHRONOLOGY,185,-810.0,37.0
1,POSITIVE_HANDOFF_GAP,315,270.0,63.0


In [25]:
# Missing segment event distribution

missing_event_analysis = (
    connecting_analysis
    .groupby("missing_segment_events")
    .agg(
        baggage_count=("bag_id", "nunique")
    )
    .reset_index()
)

missing_event_analysis["percentage"] = (
    missing_event_analysis["baggage_count"] / 500
) * 100

print("MISSING SEGMENT EVENT DISTRIBUTION:")
display(missing_event_analysis)

MISSING SEGMENT EVENT DISTRIBUTION:


,missing_segment_events,baggage_count,percentage
0,SORTED,500,100.0


In [26]:
# Consolidated connecting journey summary

connection_summary = pd.DataFrame({
    "metric": [
        "Total Connecting Baggage",
        "Missing SORTED Events",
        "Positive Handoff Chronology",
        "Invalid Handoff Chronology",
        "Airport Consistency Issues",
        "Event Duration Pattern Issues"
    ],
    "value": [
        len(connecting_analysis),
        (connecting_analysis["missing_segment_events"] == "SORTED").sum(),
        (connecting_analysis["transfer_to_load_minutes"] > 0).sum(),
        (connecting_analysis["transfer_to_load_minutes"] < 0).sum(),
        connecting_analysis["airport_consistency_issue_detected"].sum(),
        connecting_analysis["event_duration_pattern_detected"].sum()
    ]
})

print("CONNECTING JOURNEY SUMMARY:")
display(connection_summary)

CONNECTING JOURNEY SUMMARY:


,metric,value
0,Total Connecting Baggage,500
1,Missing SORTED Events,500
2,Positive Handoff Chronology,315
3,Invalid Handoff Chronology,185
4,Airport Consistency Issues,0
5,Event Duration Pattern Issues,0


In [27]:
# Dwell-time analytics

dwell_time_summary = pd.DataFrame({
    "transition": [
        "ARRIVED_TO_UNLOADED",
        "UNLOADED_TO_TRANSFER",
        "TRANSFER_TO_SECOND_LOAD"
    ],
    "average_minutes": [
        15,
        connecting_analysis["unload_to_transfer_minutes"].mean(),
        connecting_analysis["transfer_to_load_minutes"].mean()
    ],
    "median_minutes": [
        15,
        connecting_analysis["unload_to_transfer_minutes"].median(),
        connecting_analysis["transfer_to_load_minutes"].median()
    ],
    "minimum_minutes": [
        15,
        connecting_analysis["unload_to_transfer_minutes"].min(),
        connecting_analysis["transfer_to_load_minutes"].min()
    ],
    "maximum_minutes": [
        15,
        connecting_analysis["unload_to_transfer_minutes"].max(),
        connecting_analysis["transfer_to_load_minutes"].max()
    ]
})

print("DWELL-TIME ANALYTICS:")
display(dwell_time_summary.round(2))

DWELL-TIME ANALYTICS:


,transition,average_minutes,median_minutes,minimum_minutes,maximum_minutes
0,ARRIVED_TO_UNLOADED,15.0,15.0,15.0,15.0
1,UNLOADED_TO_TRANSFER,30.0,30.0,30.0,30.0
2,TRANSFER_TO_SECOND_LOAD,-129.6,270.0,-810.0,270.0


In [28]:
# Journey analytics output

journey_analytics = (
    advanced_investigation
    .groupby("journey_type")
    .agg(
        total_baggage=("bag_id", "nunique"),
        baggage_with_issues=("advanced_issue_detected", "sum")
    )
    .reset_index()
)

journey_analytics["investigation_rate"] = (
    journey_analytics["baggage_with_issues"]
    / journey_analytics["total_baggage"]
) * 100

journey_analytics["percentage_of_total"] = (
    journey_analytics["total_baggage"]
    / advanced_investigation["bag_id"].nunique()
) * 100

print("FINAL JOURNEY ANALYTICS DATASET:")
display(journey_analytics)

FINAL JOURNEY ANALYTICS DATASET:


,journey_type,total_baggage,baggage_with_issues,investigation_rate,percentage_of_total
0,CONNECTING,500,500,100.0,50.0
1,DIRECT,500,0,0.0,50.0


In [29]:
# Save journey analytics

journey_output_path = (
    processed_path / "phase13_journey_analytics.csv"
)

journey_analytics.to_csv(
    journey_output_path,
    index=False
)

print("Journey analytics saved successfully!")
print("File:", journey_output_path)
print("Shape:", journey_analytics.shape)

Journey analytics saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase13_journey_analytics.csv
Shape: (2, 5)


In [30]:
# Reload and validate journey analytics

journey_analytics_check = pd.read_csv(
    processed_path / "phase13_journey_analytics.csv"
)

print("Reloaded shape:", journey_analytics_check.shape)

print("\nSaved dataset:")
display(journey_analytics_check)

print(
    "\nValidation passed:",
    journey_analytics_check.equals(journey_analytics)
)

Reloaded shape: (2, 5)

Saved dataset:


,journey_type,total_baggage,baggage_with_issues,investigation_rate,percentage_of_total
0,CONNECTING,500,500,100.0,50.0
1,DIRECT,500,0,0.0,50.0



Validation passed: True


In [31]:
# Exception analytics output

exception_analytics = (
    exceptions
    .groupby(["exception_type", "severity", "status"])
    .agg(
        total_exceptions=("exception_id", "count"),
        unique_baggage=("bag_id", "nunique")
    )
    .reset_index()
)

print("FINAL EXCEPTION ANALYTICS DATASET:")
display(exception_analytics)

print("\nDataset shape:", exception_analytics.shape)

FINAL EXCEPTION ANALYTICS DATASET:


,exception_type,severity,status,total_exceptions,unique_baggage
0,HANDOFF_TIMING_INCONSISTENCY,HIGH,OPEN,184,184
1,HANDOFF_TIMING_INCONSISTENCY,HIGH,RESOLVED,1,1
2,MISSING_SEGMENT_EVENT,MEDIUM,OPEN,500,500



Dataset shape: (3, 5)


In [32]:
# Save exception analytics

exception_output_path = (
    processed_path / "phase13_exception_analytics.csv"
)

exception_analytics.to_csv(
    exception_output_path,
    index=False
)

print("Exception analytics saved successfully!")
print("File:", exception_output_path)
print("Shape:", exception_analytics.shape)

Exception analytics saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase13_exception_analytics.csv
Shape: (3, 5)


In [33]:
# Reload and validate exception analytics

exception_analytics_check = pd.read_csv(
    processed_path / "phase13_exception_analytics.csv"
)

print("Reloaded shape:", exception_analytics_check.shape)

print("\nSaved exception analytics:")
display(exception_analytics_check)

print(
    "\nValidation passed:",
    exception_analytics_check.equals(exception_analytics)
)

print(
    "\nTotal exception records:",
    exception_analytics_check["total_exceptions"].sum()
)

Reloaded shape: (3, 5)

Saved exception analytics:


,exception_type,severity,status,total_exceptions,unique_baggage
0,HANDOFF_TIMING_INCONSISTENCY,HIGH,OPEN,184,184
1,HANDOFF_TIMING_INCONSISTENCY,HIGH,RESOLVED,1,1
2,MISSING_SEGMENT_EVENT,MEDIUM,OPEN,500,500



Validation passed: True

Total exception records: 685


In [34]:
# Airport analytics output

airport_analytics = (
    baggage_scan_event
    .groupby("airport_code")
    .agg(
        total_scan_events=("event_id", "count"),
        unique_baggage=("bag_id", "nunique"),
        event_types=("event_type", "nunique")
    )
    .reset_index()
)

airport_analytics["percentage_of_total_scans"] = (
    airport_analytics["total_scan_events"]
    / airport_analytics["total_scan_events"].sum()
) * 100

airport_analytics = airport_analytics.sort_values(
    by="total_scan_events",
    ascending=False
).reset_index(drop=True)

print("FINAL AIRPORT ANALYTICS DATASET:")
display(airport_analytics.round(2))

print("\nDataset shape:", airport_analytics.shape)
print(
    "Total scan events:",
    airport_analytics["total_scan_events"].sum()
)

FINAL AIRPORT ANALYTICS DATASET:


,airport_code,total_scan_events,unique_baggage,event_types,percentage_of_total_scans
0,MAA,1676,419,4,18.62
1,LHR,1317,439,3,14.63
2,DXB,1088,290,8,12.09
3,DEL,1065,278,8,11.83
4,DOH,1033,268,8,11.48
5,SIN,849,232,8,9.43
6,FRA,660,206,7,7.33
7,BOM,624,156,4,6.93
8,JFK,480,160,3,5.33
9,BLR,208,52,4,2.31



Dataset shape: (10, 5)
Total scan events: 9000


In [35]:
# Save airport analytics

airport_output_path = (
    processed_path / "phase13_airport_analytics.csv"
)

airport_analytics.to_csv(
    airport_output_path,
    index=False
)

print("Airport analytics saved successfully!")
print("File:", airport_output_path)
print("Shape:", airport_analytics.shape)

Airport analytics saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase13_airport_analytics.csv
Shape: (10, 5)


In [36]:
# Reload and validate airport analytics

airport_analytics_check = pd.read_csv(
    processed_path / "phase13_airport_analytics.csv"
)

print("Reloaded shape:", airport_analytics_check.shape)

print("\nSaved airport analytics:")
display(airport_analytics_check)

print(
    "\nValidation passed:",
    airport_analytics_check.equals(airport_analytics)
)

print(
    "\nTotal scan events:",
    airport_analytics_check["total_scan_events"].sum()
)

Reloaded shape: (10, 5)

Saved airport analytics:


,airport_code,total_scan_events,unique_baggage,event_types,percentage_of_total_scans
0,MAA,1676,419,4,18.622222
1,LHR,1317,439,3,14.633333
2,DXB,1088,290,8,12.088889
3,DEL,1065,278,8,11.833333
4,DOH,1033,268,8,11.477778
5,SIN,849,232,8,9.433333
6,FRA,660,206,7,7.333333
7,BOM,624,156,4,6.933333
8,JFK,480,160,3,5.333333
9,BLR,208,52,4,2.311111



Validation passed: True

Total scan events: 9000


In [37]:
# Final flight analytics dataset

flight_analytics = flight_enriched.copy()

flight_analytics["percentage_of_total_assignments"] = (
    flight_analytics["total_assignments"]
    / flight_analytics["total_assignments"].sum()
) * 100

flight_analytics = flight_analytics.sort_values(
    by="total_baggage",
    ascending=False
).reset_index(drop=True)

print("FINAL FLIGHT ANALYTICS DATASET:")
display(flight_analytics.head(10))

print("\nDataset shape:", flight_analytics.shape)

print(
    "\nTotal baggage assignments:",
    flight_analytics["total_assignments"].sum()
)

print(
    "\nFlights represented:",
    flight_analytics["flight_id"].nunique()
)

FINAL FLIGHT ANALYTICS DATASET:


,flight_id,flight_number,source,destination,total_baggage,total_assignments,percentage_of_total_assignments
0,FL0051,6E5268,DEL,LHR,19,19,1.266667
1,FL0041,BA4474,MAA,DEL,19,19,1.266667
2,FL0101,6E2160,DXB,JFK,17,17,1.133333
3,FL0102,EK7887,DXB,JFK,17,17,1.133333
4,FL0057,EK4439,DEL,LHR,16,16,1.066667
5,FL0092,QR8105,BOM,DXB,16,16,1.066667
6,FL0020,6E5614,DXB,LHR,16,16,1.066667
7,FL0047,BA9225,MAA,DEL,15,15,1.000000
8,FL0079,AI4908,DOH,FRA,15,15,1.000000
9,FL0066,AI871,MAA,DOH,15,15,1.000000



Dataset shape: (179, 7)

Total baggage assignments: 1500

Flights represented: 179


In [38]:
# Save flight analytics

flight_output_path = (
    processed_path / "phase13_flight_analytics.csv"
)

flight_analytics.to_csv(
    flight_output_path,
    index=False
)

print("Flight analytics saved successfully!")
print("File:", flight_output_path)
print("Shape:", flight_analytics.shape)

Flight analytics saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase13_flight_analytics.csv
Shape: (179, 7)


In [39]:
# Reload and validate flight analytics

flight_analytics_check = pd.read_csv(
    processed_path / "phase13_flight_analytics.csv"
)

print("Reloaded shape:", flight_analytics_check.shape)

print("\nSaved flight analytics:")
display(flight_analytics_check.head(10))

print(
    "\nValidation passed:",
    flight_analytics_check.equals(flight_analytics)
)

print(
    "\nTotal baggage assignments:",
    flight_analytics_check["total_assignments"].sum()
)

print(
    "\nFlights represented:",
    flight_analytics_check["flight_id"].nunique()
)

Reloaded shape: (179, 7)

Saved flight analytics:


,flight_id,flight_number,source,destination,total_baggage,total_assignments,percentage_of_total_assignments
0,FL0051,6E5268,DEL,LHR,19,19,1.266667
1,FL0041,BA4474,MAA,DEL,19,19,1.266667
2,FL0101,6E2160,DXB,JFK,17,17,1.133333
3,FL0102,EK7887,DXB,JFK,17,17,1.133333
4,FL0057,EK4439,DEL,LHR,16,16,1.066667
5,FL0092,QR8105,BOM,DXB,16,16,1.066667
6,FL0020,6E5614,DXB,LHR,16,16,1.066667
7,FL0047,BA9225,MAA,DEL,15,15,1.000000
8,FL0079,AI4908,DOH,FRA,15,15,1.000000
9,FL0066,AI871,MAA,DOH,15,15,1.000000



Validation passed: True

Total baggage assignments: 1500

Flights represented: 179


In [40]:
# Final connection analytics dataset

connection_analytics = connecting_analysis[
    [
        "bag_id",
        "missing_segment_event_count",
        "missing_segment_events",
        "unload_to_transfer_minutes",
        "transfer_to_load_minutes",
        "handoff_chronology_valid",
        "handoff_issue",
        "airport_consistent",
        "missing_segment_event_detected",
        "handoff_timing_inconsistency_detected",
        "advanced_detection_status",
        "advanced_severity"
    ]
].copy()

print("FINAL CONNECTION ANALYTICS DATASET:")
display(connection_analytics.head(10))

print("\nDataset shape:", connection_analytics.shape)

print(
    "\nUnique connecting baggage:",
    connection_analytics["bag_id"].nunique()
)

FINAL CONNECTION ANALYTICS DATASET:


,bag_id,missing_segment_event_count,missing_segment_events,unload_to_transfer_minutes,transfer_to_load_minutes,handoff_chronology_valid,handoff_issue,airport_consistent,missing_segment_event_detected,handoff_timing_inconsistency_detected,advanced_detection_status,advanced_severity
500,BAG00501,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH
501,BAG00502,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
502,BAG00503,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
503,BAG00504,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
504,BAG00505,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
505,BAG00506,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH
506,BAG00507,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
507,BAG00508,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
508,BAG00509,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH
509,BAG00510,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH



Dataset shape: (500, 12)

Unique connecting baggage: 500


In [41]:
# Save connection analytics

connection_output_path = (
    processed_path / "phase13_connection_analytics.csv"
)

connection_analytics.to_csv(
    connection_output_path,
    index=False
)

print("Connection analytics saved successfully!")
print("File:", connection_output_path)
print("Shape:", connection_analytics.shape)

Connection analytics saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase13_connection_analytics.csv
Shape: (500, 12)


In [42]:
# Reload and validate connection analytics

connection_analytics_check = pd.read_csv(
    processed_path / "phase13_connection_analytics.csv"
)

print("Reloaded shape:", connection_analytics_check.shape)

print(
    "\nValidation passed:",
    connection_analytics_check.equals(connection_analytics)
)

print(
    "\nUnique baggage IDs:",
    connection_analytics_check["bag_id"].nunique()
)

print("\nHandoff chronology distribution:")
display(
    connection_analytics_check["handoff_chronology_valid"]
    .value_counts()
)

print("\nInvestigation status distribution:")
display(
    connection_analytics_check["advanced_detection_status"]
    .value_counts()
)

Reloaded shape: (500, 12)

Validation passed: False

Unique baggage IDs: 500

Handoff chronology distribution:


handoff_chronology_valid
True     315
False    185
Name: count, dtype: int64


Investigation status distribution:


advanced_detection_status
MISSING_EVENT_ONLY                        315
MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE    185
Name: count, dtype: int64

In [43]:
# Identify differences between original and reloaded datasets

print("ORIGINAL DATA TYPES:")
display(connection_analytics.dtypes)

print("\nRELOADED DATA TYPES:")
display(connection_analytics_check.dtypes)

print("\nCOLUMN-WISE VALIDATION:")

for column in connection_analytics.columns:
    same = connection_analytics[column].equals(
        connection_analytics_check[column]
    )

    print(f"{column}: {same}")

print("\nORIGINAL DATASET:")
display(connection_analytics.head(3))

print("\nRELOADED DATASET:")
display(connection_analytics_check.head(3))

ORIGINAL DATA TYPES:


bag_id                                       str
missing_segment_event_count                int64
missing_segment_events                       str
unload_to_transfer_minutes               float64
transfer_to_load_minutes                 float64
handoff_chronology_valid                  object
handoff_issue                                str
airport_consistent                        object
missing_segment_event_detected              bool
handoff_timing_inconsistency_detected       bool
advanced_detection_status                    str
advanced_severity                            str
dtype: object


RELOADED DATA TYPES:


bag_id                                       str
missing_segment_event_count                int64
missing_segment_events                       str
unload_to_transfer_minutes               float64
transfer_to_load_minutes                 float64
handoff_chronology_valid                    bool
handoff_issue                                str
airport_consistent                          bool
missing_segment_event_detected              bool
handoff_timing_inconsistency_detected       bool
advanced_detection_status                    str
advanced_severity                            str
dtype: object


COLUMN-WISE VALIDATION:
bag_id: False
missing_segment_event_count: False
missing_segment_events: False
unload_to_transfer_minutes: False
transfer_to_load_minutes: False
handoff_chronology_valid: False
handoff_issue: False
airport_consistent: False
missing_segment_event_detected: False
handoff_timing_inconsistency_detected: False
advanced_detection_status: False
advanced_severity: False

ORIGINAL DATASET:


,bag_id,missing_segment_event_count,missing_segment_events,unload_to_transfer_minutes,transfer_to_load_minutes,handoff_chronology_valid,handoff_issue,airport_consistent,missing_segment_event_detected,handoff_timing_inconsistency_detected,advanced_detection_status,advanced_severity
500,BAG00501,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH
501,BAG00502,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
502,BAG00503,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM



RELOADED DATASET:


,bag_id,missing_segment_event_count,missing_segment_events,unload_to_transfer_minutes,transfer_to_load_minutes,handoff_chronology_valid,handoff_issue,airport_consistent,missing_segment_event_detected,handoff_timing_inconsistency_detected,advanced_detection_status,advanced_severity
0,BAG00501,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH
1,BAG00502,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
2,BAG00503,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM


In [44]:
# Normalize the original index for comparison

connection_original_check = (
    connection_analytics.reset_index(drop=True).copy()
)

# Normalize boolean column data types

boolean_columns = [
    "handoff_chronology_valid",
    "airport_consistent"
]

for column in boolean_columns:
    connection_original_check[column] = (
        connection_original_check[column].astype(bool)
    )

    connection_analytics_check[column] = (
        connection_analytics_check[column].astype(bool)
    )

# Compare both datasets

print(
    "Validation passed:",
    connection_original_check.equals(connection_analytics_check)
)

print("\nOriginal missing values:")
display(connection_original_check.isna().sum())

print("\nReloaded missing values:")
display(connection_analytics_check.isna().sum())

Validation passed: True

Original missing values:


bag_id                                   0
missing_segment_event_count              0
missing_segment_events                   0
unload_to_transfer_minutes               0
transfer_to_load_minutes                 0
handoff_chronology_valid                 0
handoff_issue                            0
airport_consistent                       0
missing_segment_event_detected           0
handoff_timing_inconsistency_detected    0
advanced_detection_status                0
advanced_severity                        0
dtype: int64


Reloaded missing values:


bag_id                                   0
missing_segment_event_count              0
missing_segment_events                   0
unload_to_transfer_minutes               0
transfer_to_load_minutes                 0
handoff_chronology_valid                 0
handoff_issue                            0
airport_consistent                       0
missing_segment_event_detected           0
handoff_timing_inconsistency_detected    0
advanced_detection_status                0
advanced_severity                        0
dtype: int64

In [45]:
print("DWELL TIME SUMMARY:")
display(dwell_time_summary)

print("\nShape:", dwell_time_summary.shape)
print("\nColumns:", dwell_time_summary.columns.tolist())

DWELL TIME SUMMARY:


,transition,average_minutes,median_minutes,minimum_minutes,maximum_minutes
0,ARRIVED_TO_UNLOADED,15.0,15.0,15.0,15.0
1,UNLOADED_TO_TRANSFER,30.0,30.0,30.0,30.0
2,TRANSFER_TO_SECOND_LOAD,-129.6,270.0,-810.0,270.0



Shape: (3, 5)

Columns: ['transition', 'average_minutes', 'median_minutes', 'minimum_minutes', 'maximum_minutes']


In [46]:
# Save dwell-time analytics

dwell_output_path = (
    processed_path / "phase13_dwell_time_analytics.csv"
)

dwell_time_summary.to_csv(
    dwell_output_path,
    index=False
)

print("Dwell-time analytics saved successfully!")
print("File:", dwell_output_path)
print("Shape:", dwell_time_summary.shape)

Dwell-time analytics saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase13_dwell_time_analytics.csv
Shape: (3, 5)


In [47]:
# Reload and validate dwell-time analytics

dwell_time_check = pd.read_csv(
    processed_path / "phase13_dwell_time_analytics.csv"
)

print("Reloaded shape:", dwell_time_check.shape)

print("\nSaved dataset:")
display(dwell_time_check)

print(
    "\nValidation passed:",
    dwell_time_check.reset_index(drop=True).equals(
        dwell_time_summary.reset_index(drop=True)
    )
)

print(
    "\nMissing values:",
    dwell_time_check.isna().sum().sum()
)

Reloaded shape: (3, 5)

Saved dataset:


,transition,average_minutes,median_minutes,minimum_minutes,maximum_minutes
0,ARRIVED_TO_UNLOADED,15.0,15.0,15.0,15.0
1,UNLOADED_TO_TRANSFER,30.0,30.0,30.0,30.0
2,TRANSFER_TO_SECOND_LOAD,-129.6,270.0,-810.0,270.0



Validation passed: True

Missing values: 0


In [48]:
# Final Phase 13 output inventory

phase13_files = {
    "phase13_journey_analytics.csv": (2, 5),
    "phase13_exception_analytics.csv": (3, 5),
    "phase13_airport_analytics.csv": (10, 5),
    "phase13_flight_analytics.csv": (179, 7),
    "phase13_connection_analytics.csv": (500, 12),
    "phase13_dwell_time_analytics.csv": (3, 5)
}

inventory = []

for filename, expected_shape in phase13_files.items():

    file_path = processed_path / filename

    if file_path.exists():
        df = pd.read_csv(file_path)

        inventory.append({
            "file_name": filename,
            "exists": True,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "expected_shape": expected_shape,
            "shape_valid": df.shape == expected_shape,
            "missing_values": df.isna().sum().sum()
        })

    else:
        inventory.append({
            "file_name": filename,
            "exists": False,
            "rows": 0,
            "columns": 0,
            "expected_shape": expected_shape,
            "shape_valid": False,
            "missing_values": None
        })

phase13_inventory = pd.DataFrame(inventory)

display(phase13_inventory)

print("\nAll six files exist:",
      phase13_inventory["exists"].all())

print("All shapes valid:",
      phase13_inventory["shape_valid"].all())

print("Total analytical output rows:",
      phase13_inventory["rows"].sum())

print("Total missing values:",
      phase13_inventory["missing_values"].sum())

,file_name,exists,rows,columns,expected_shape,shape_valid,missing_values
0,phase13_journey_analytics.csv,True,2,5,"(2, 5)",True,0
1,phase13_exception_analytics.csv,True,3,5,"(3, 5)",True,0
2,phase13_airport_analytics.csv,True,10,5,"(10, 5)",True,0
3,phase13_flight_analytics.csv,True,179,7,"(179, 7)",True,0
4,phase13_connection_analytics.csv,True,500,12,"(500, 12)",True,0
5,phase13_dwell_time_analytics.csv,True,3,5,"(3, 5)",True,0



All six files exist: True
All shapes valid: True
Total analytical output rows: 697
Total missing values: 0
